# [스프린트 미션 3] Baseline: 캐글 타이타닉 생존자 예측

이 노트북은 미션 3을 시작할 때 참고할 수 있는 Baseline 코드입니다.

> **주의**: Baseline 코드는 참고 자료일 뿐입니다. 그대로 제출하기보다는, 이 코드를 출발점 삼아 전처리·피처·모델을 자유롭게 바꿔서 리더보드 점수를 끌어올려 보세요.

이 노트북에서는 두 가지를 특히 신경 써서 만들었어요.
1. **실험 기록 자동화**: 모델을 하나 학습시킬 때마다 `run_experiment()` 한 줄만 호출하면, 그 실험의 하이퍼파라미터와 train 정확도·검증 정확도·교차 검증 점수가 표(DataFrame)에 자동으로 쌓입니다. train과 검증 점수의 차이를 보면 과적합 여부까지 표에서 바로 읽을 수 있어요.
2. **모델 진단 시각화**: Confusion Matrix와 Feature Importance를 함수 하나로 바로 확인할 수 있게 만들어 뒀어요. 모델이 어디서 틀리는지, 어떤 피처를 많이 참고하는지를 눈으로 보면서 다음 실험 방향을 정할 수 있습니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("설정 완료")

In [ ]:
# matplotlib 한글 깨짐 방지 (Colab)
!apt-get update -qq
!apt-get install -y -qq fonts-nanum*
!rm -rf ~/.cache/matplotlib

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings
warnings.filterwarnings(action="ignore")

path = "/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf"  # 나눔바른고딕
fm.fontManager.addfont(path)
plt.rc("font", family=fm.FontProperties(fname=path).get_name())

## 1. Kaggle CLI로 데이터 받기
[대회 페이지](https://www.kaggle.com/c/titanic)에서 **Join Competition**을 먼저 누른 뒤 진행하세요. (참가하지 않으면 다운로드에서 403 에러가 납니다)
1. 캐글 프로필 사진 \> Settings \> API에서 토큰을 생성하고, `KGAT_`로 시작하는 문자열을 복사해 두세요.
2. Colab 왼쪽 사이드바의 🔑(보안 비밀) 탭에서 이름은 `KAGGLE_API_TOKEN`, 값은 복사해 둔 토큰으로 저장하고, "노트북 액세스"를 켜 주세요. 한 번 저장해 두면 런타임이 초기화되거나 노트북을 새로 만들어도 계속 쓸 수 있습니다.

> 토큰을 노트북 셀에 직접 붙여넣지 마세요. 제출한 노트북에 토큰이 그대로 남습니다.

In [ ]:
%pip install -q --upgrade kaggle  # KGAT_ 토큰 인증은 최신 버전에서 지원됩니다.

import os
from google.colab import userdata

# Colab 보안 비밀에 저장해 둔 토큰을 환경 변수로 등록합니다.
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")

!kaggle competitions download -c titanic
!unzip -q titanic.zip -d data
!ls data

In [ ]:
train_df = pd.read_csv("data/train.csv")
test_df = pd.read_csv("data/test.csv")

print("train:", train_df.shape, "| test:", test_df.shape)
train_df.head()

## 2. 실험 기록 도구 만들기
앞으로 어떤 모델을 실험하든, 아래 `run_experiment()`로 학습하면 결과가 `experiment_log`에 자동으로 쌓입니다.
- **train 정확도 vs 검증 정확도**: 두 값의 차이가 크면 과적합을 의심할 수 있어요.
- **CV 평균·표준편차**: 데이터를 나누는 방식에 따라 점수가 얼마나 흔들리는지 보여줍니다.
- **하이퍼파라미터**: `model.get_params()`로 자동 추출하므로 손으로 다시 적을 필요가 없습니다. 모델마다 파라미터 종류가 달라도, 표를 만들 때 없는 값은 빈칸(NaN)으로 채워집니다.

In [ ]:
experiment_log = []  # 실험 결과가 쌓이는 곳


def run_experiment(name, pipeline, X_tr, y_tr, X_va, y_va, cv=5):
    """모델을 학습시키고, 성능과 하이퍼파라미터를 experiment_log에 자동으로 기록합니다.

    반환값은 지금까지의 전체 실험 기록을 검증 정확도 순으로 정렬한 DataFrame입니다.
    """
    pipeline.fit(X_tr, y_tr)

    cv_scores = cross_val_score(pipeline, X_tr, y_tr, cv=cv, scoring="accuracy")
    row = {
        "실험명": name,
        "train 정확도": pipeline.score(X_tr, y_tr),
        "검증 정확도": pipeline.score(X_va, y_va),
        "CV 평균": cv_scores.mean(),
        "CV 표준편차": cv_scores.std(),
    }
    row.update(pipeline.named_steps["model"].get_params())
    experiment_log.append(row)

    log_df = pd.DataFrame(experiment_log)
    return log_df.sort_values("검증 정확도", ascending=False).reset_index(drop=True)


def show_log(columns=None):
    """실험 기록을 보여줍니다. columns를 지정하면 그 컬럼들만 골라서 봅니다.

    예: show_log(["실험명", "검증 정확도", "n_estimators", "max_depth"])
    """
    log_df = pd.DataFrame(experiment_log).sort_values("검증 정확도", ascending=False).reset_index(drop=True)
    return log_df[columns] if columns else log_df

## 3. 데이터 훑어보기
결측치와 생존율 패턴을 확인해 둡니다. Age는 약 20%, Cabin은 약 77%가 비어 있어서 처리 방식을 다르게 가져가야 해요.

In [ ]:
print("결측치 개수")
print(train_df.isnull().sum())
print()
print("전체 생존율:", train_df["Survived"].mean().round(3))

In [ ]:
# 성별과 티켓 등급에 따라 생존율이 크게 갈립니다.
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
train_df.groupby("Sex")["Survived"].mean().plot.bar(ax=axes[0], title="성별 생존율", rot=0)
train_df.groupby("Pclass")["Survived"].mean().plot.bar(ax=axes[1], title="티켓 등급별 생존율", rot=0)
plt.tight_layout()
plt.show()

## 4. 피처 엔지니어링
가이드라인에서 언급한 파생 변수들을 만드는 함수입니다. `train_df`와 `test_df`에 똑같이 적용해야, 두 데이터의 컬럼 구성이 어긋나지 않습니다.

In [ ]:
def add_features(df):
    """Name, SibSp/Parch, Cabin으로부터 파생 변수를 만듭니다."""
    df = df.copy()

    # Name에서 호칭 추출: "Braund, Mr. Owen Harris" -> "Mr"
    df["Title"] = df["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
    df["Title"] = df["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
    common = ["Mr", "Mrs", "Miss", "Master"]
    df["Title"] = df["Title"].where(df["Title"].isin(common), "Rare")

    # 가족 규모와 혼자 탑승 여부
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1)

    # Cabin은 77%가 결측이라 값을 채우는 대신, 정보가 있는지 여부를 변수로 씁니다.
    df["HasCabin"] = df["Cabin"].notnull()

    return df


train_fe = add_features(train_df)
test_fe = add_features(test_df)

train_fe.groupby("Title")["Survived"].agg(["mean", "count"])

## 5. 전처리 파이프라인 구성
수치형 컬럼은 결측치를 중앙값으로 채우고 표준화하고, 범주형 컬럼은 최빈값으로 채우고 원-핫 인코딩합니다.
`Pipeline`으로 전처리와 모델을 묶어 두면, 교차 검증을 할 때도 train 기준으로만 전처리 통계(중앙값, 최빈값 등)를 계산하게 되어 데이터 누수를 막을 수 있습니다.

In [ ]:
categorical_features = ["Pclass", "Sex", "Embarked", "Title", "IsAlone", "HasCabin"]
numeric_features = ["Age", "Fare", "SibSp", "Parch", "FamilySize"]


numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features),
])

X = train_fe[numeric_features + categorical_features]
y = train_fe["Survived"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("train:", X_train.shape, "| val:", X_val.shape)

## 6. 모델 진단 시각화 도구
모델을 학습시킬 때마다 반복해서 쓸 두 가지 함수를 만들어 둡니다.
- `plot_confusion(...)`: 어떤 클래스를 얼마나 헷갈렸는지 한눈에 보여줍니다.
- `plot_feature_importance(...)`: 모델이 어떤 피처를 많이 참고했는지 보여줍니다. 트리 계열 모델은 `feature_importances_`를, 로지스틱 회귀는 `coef_`를 사용합니다.

In [ ]:
def plot_confusion(pipeline, X, y, title):
    ConfusionMatrixDisplay.from_estimator(
        pipeline, X, y,
        display_labels=["사망(0)", "생존(1)"],
        cmap="Blues",
    )
    plt.title(title)
    plt.show()
    print(classification_report(y, pipeline.predict(X), target_names=["사망(0)", "생존(1)"]))


def plot_feature_importance(pipeline, title, top_n=15):
    model = pipeline.named_steps["model"]
    feature_names = pipeline.named_steps["preprocessor"].get_feature_names_out()

    if hasattr(model, "feature_importances_"):
        importances = model.feature_importances_
    elif hasattr(model, "coef_"):
        importances = np.abs(model.coef_[0])
    else:
        print(f"{title}: 이 모델은 feature_importances_나 coef_가 없어서 중요도를 그릴 수 없습니다.")
        return

    order = np.argsort(importances)[-top_n:]
    plt.figure(figsize=(7, 5))
    plt.barh(range(len(order)), importances[order])
    plt.yticks(range(len(order)), feature_names[order])
    plt.xlabel("중요도")
    plt.title(title)
    plt.tight_layout()
    plt.show()

## 7. 실험 1: 로지스틱 회귀 (베이스라인)

In [ ]:
log_reg_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

run_experiment("로지스틱 회귀", log_reg_pipe, X_train, y_train, X_val, y_val)[
    ["실험명", "train 정확도", "검증 정확도", "CV 평균"]
]

In [ ]:
plot_confusion(log_reg_pipe, X_val, y_val, "로지스틱 회귀")
plot_feature_importance(log_reg_pipe, "로지스틱 회귀: 계수 크기")

## 8. 실험 2: 결정트리
`max_depth`를 제한하지 않으면 train 정확도만 치솟는 과적합이 일어납니다. 실험 기록에서 train과 검증 정확도의 차이를 비교해 보세요.

In [ ]:
tree_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE)),
])

run_experiment("결정트리", tree_pipe, X_train, y_train, X_val, y_val)[
    ["실험명", "train 정확도", "검증 정확도", "CV 평균"]
]

In [ ]:
plot_confusion(tree_pipe, X_val, y_val, "결정트리")
plot_feature_importance(tree_pipe, "결정트리: 피처 중요도")

## 9. 실험 3: 랜덤 포레스트

In [ ]:
rf_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(n_estimators=200, max_depth=6, random_state=RANDOM_STATE)),
])

run_experiment("랜덤 포레스트", rf_pipe, X_train, y_train, X_val, y_val)[
    ["실험명", "train 정확도", "검증 정확도", "CV 평균"]
]

In [ ]:
plot_confusion(rf_pipe, X_val, y_val, "랜덤 포레스트")
plot_feature_importance(rf_pipe, "랜덤 포레스트: 피처 중요도")

## 10. 실험 4: 그레이디언트 부스팅

In [ ]:
gb_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GradientBoostingClassifier(n_estimators=200, max_depth=3, random_state=RANDOM_STATE)),
])

run_experiment("그레이디언트 부스팅", gb_pipe, X_train, y_train, X_val, y_val)[
    ["실험명", "train 정확도", "검증 정확도", "CV 평균"]
]

In [ ]:
plot_confusion(gb_pipe, X_val, y_val, "그레이디언트 부스팅")
plot_feature_importance(gb_pipe, "그레이디언트 부스팅: 피처 중요도")

## 11. 실험 기록 한눈에 보기
지금까지의 실험을 정렬해서 비교합니다. train 정확도와 검증 정확도의 차이가 큰 모델은 과적합을 의심해 보세요.
하이퍼파라미터까지 포함한 전체 표가 궁금하면 `show_log()`를 그대로 호출하고, 관심 있는 컬럼만 보고 싶으면 컬럼 목록을 넘기면 됩니다.
예: `show_log(["실험명", "검증 정확도", "n_estimators", "max_depth"])`

In [ ]:
log_df = show_log(["실험명", "train 정확도", "검증 정확도", "CV 평균", "CV 표준편차"])
log_df

In [ ]:
x = np.arange(len(log_df))
plt.figure(figsize=(8, 4))
plt.bar(x - 0.2, log_df["train 정확도"], width=0.4, label="train")
plt.bar(x + 0.2, log_df["검증 정확도"], width=0.4, label="검증")
plt.xticks(x, log_df["실험명"], rotation=15)
plt.ylim(0.7, 1.0)
plt.ylabel("정확도")
plt.title("모델별 train vs 검증 정확도")
plt.legend()
plt.tight_layout()
plt.show()

## 12. 그리드 서치로 튜닝하기
가장 성능이 좋았던 모델을 골라서 하이퍼파라미터를 더 탐색해 봅시다. 여기서는 랜덤 포레스트로 진행합니다.
`GridSearchCV`가 시도한 모든 조합은 `cv_results_`에 이미 표 형태로 정리되어 있습니다. 우리가 직접 만든 `experiment_log`와 같은 아이디어를, 사이킷런은 내부적으로 이미 하고 있었던 거예요. 이 결과를 우리 실험 기록에도 합쳐 봅니다.

In [ ]:
param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [4, 6, 8, None],
    "model__min_samples_leaf": [1, 3, 5],
}

grid_search = GridSearchCV(rf_pipe, param_grid, cv=5, scoring="accuracy", n_jobs=-1)
grid_search.fit(X_train, y_train)

print("최적 파라미터:", grid_search.best_params_)
print("최적 CV 점수:", grid_search.best_score_.round(4))

grid_results = pd.DataFrame(grid_search.cv_results_)
grid_results[["params", "mean_test_score", "std_test_score"]].sort_values(
    "mean_test_score", ascending=False
).head()

In [ ]:
# 그리드 서치 결과를 우리 실험 기록에도 합쳐 둡니다.
best_model = grid_search.best_estimator_

for _, row in grid_results.iterrows():
    entry = {
        "실험명": "랜덤 포레스트 (GridSearch)",
        "CV 평균": row["mean_test_score"],
        "CV 표준편차": row["std_test_score"],
    }
    entry.update(row["params"])
    experiment_log.append(entry)

# 최적 모델의 train/검증 정확도도 별도 실험으로 기록해 둡니다.
run_experiment("랜덤 포레스트 (최적)", best_model, X_train, y_train, X_val, y_val)[
    ["실험명", "train 정확도", "검증 정확도", "CV 평균"]
].head()

## 13. 최종 모델 평가

In [ ]:
plot_confusion(best_model, X_val, y_val, "최종 모델 (튜닝된 랜덤 포레스트)")
plot_feature_importance(best_model, "최종 모델: 피처 중요도")

## 14. 제출 파일 만들고 캐글에 제출하기
전체 훈련 데이터로 최종 모델을 다시 학습시키고, test 데이터에 대한 예측으로 `submission.csv`를 만듭니다.
제출 파일은 `PassengerId`와 `Survived`(0 또는 1) 두 컬럼이어야 합니다.

In [ ]:
best_model.fit(X, y)  # 검증 데이터까지 포함해서 전체 train으로 다시 학습

X_test = test_fe[numeric_features + categorical_features]
predictions = best_model.predict(X_test)

submission = pd.DataFrame({
    "PassengerId": test_fe["PassengerId"],
    "Survived": predictions.astype(int),
})
submission.to_csv("submission.csv", index=False)
print(submission.shape)
submission.head()

In [ ]:
# ⚠️ 이 셀은 실행할 때마다 하루 제출 기회(10번)를 1번 사용합니다.
# [런타임 > 모두 실행]에 휩쓸려 제출되지 않도록 주석으로 막아 뒀어요.
# 진짜 제출할 준비가 되면 아래 한 줄의 주석(#)을 지우고, 이 셀만 실행하세요.
# -m 뒤의 메시지에는 어떤 실험이었는지 적어 두세요.

# !kaggle competitions submit -c titanic -f submission.csv -m "baseline: random forest + grid search"

# 제출 이력과 점수 확인은 제출 기회를 쓰지 않으니 자유롭게 실행해도 됩니다.
!kaggle competitions submissions -c titanic

In [ ]:
# CLI가 잘 동작하지 않으면 파일을 내려받아 대회 페이지의 Submit Predictions 버튼으로 직접 업로드하세요.
# from google.colab import files
# files.download("submission.csv")

## 15. 여기서부터는 여러분 차례입니다
Baseline은 기본적인 전처리와 네 가지 모델만 실험했습니다. 아래 방향들을 시도하며 리더보드 점수를 끌어올려 보세요.
- Age 결측을 중앙값 대신 호칭(Title)별 중앙값으로 채워 보기
- `Cabin`의 첫 글자(갑판)를 변수로 만들어 보기
- Fare에 로그 변환을 적용하거나 구간으로 나눠 보기
- 여러 모델을 결합하는 스태킹, 보팅 앙상블 시도해 보기
- `experiment_log`를 CSV로 저장해서, 실험이 많아져도 이전 결과를 잃지 않고 이어서 비교하기 (`pd.DataFrame(experiment_log).to_csv("my_experiments.csv", index=False)`)

리더보드 제출과 `03_{팀명}_{성함}.ipynb` 노트북 제출, 둘 다 잊지 마세요!